In [2]:
"""
generate_html.py
Reads sw_solution.json (produced by export_sw_solution.m) and generates
a self-contained visual.html with the SW2007 impulse-response explorer.

Usage:
    python generate_html.py
Output:
    visual.html  — open directly in browser or host on GitHub Pages
"""

import json
import numpy as np
import sys

# ── Load solution ──────────────────────────────────────────────────────────
with open("sw_solution.json", "r") as f:
    sol = json.load(f)

# Reconstruct numpy matrices from Dynare's JSON export
ghx = np.array(sol["ghx"])          # n_endo x n_states, DR order
ghu = np.array(sol["ghu"])          # n_endo x n_exo,   DR order
state_rows = [x - 1 for x in sol["state_rows_dr"]]  # 0-indexed (MATLAB is 1-indexed)
var_names_dr = sol["var_names_dr"]
shock_names  = sol["shock_names"]
params       = sol["params"]

n_endo, n_states = ghx.shape
n_exo = ghu.shape[1]

print(f"ghx: {ghx.shape}, ghu: {ghu.shape}")
print(f"State rows (0-idx): {state_rows}")

# ── Identify variable positions in DR order ────────────────────────────────
def dr_idx(name):
    return var_names_dr.index(name)

y_i   = dr_idx("y")
yf_i  = dr_idx("yf")
pi_i  = dr_idx("pinf")
r_i   = dr_idx("r")
rrf_i = dr_idx("rrf")
c_i   = dr_idx("c")
inv_i = dr_idx("inve")
lab_i = dr_idx("lab")
w_i   = dr_idx("w")

# Shock indices
def exo_idx(name):
    return shock_names.index(name)

shock_config = [
    {"key": "ea",   "label": "Technology",           "idx": exo_idx("ea")},
    {"key": "eb",   "label": "Risk premium (demand)", "idx": exo_idx("eb")},
    {"key": "eg",   "label": "Govt spending",         "idx": exo_idx("eg")},
    {"key": "eqs",  "label": "Investment tech",       "idx": exo_idx("eqs")},
    {"key": "em",   "label": "Monetary policy",       "idx": exo_idx("em")},
    {"key": "epinf","label": "Price markup",           "idx": exo_idx("epinf")},
    {"key": "ew",   "label": "Wage markup",            "idx": exo_idx("ew")},
]

# ── Compute IRFs in Python for verification ────────────────────────────────
T = 60

def compute_irf(shock_idx, size=1.0):
    """Propagate IRF using Dynare's ghx/ghu."""
    states  = np.zeros(n_endo)
    states += ghu[:, shock_idx] * size
    path = np.zeros((T, n_endo))
    path[0] = states
    for t in range(1, T):
        state_vec = path[t-1, state_rows]  # extract state variables
        path[t]   = ghx @ state_vec
    return path

# Sanity checks
print("\nSanity checks:")
irf_tech = compute_irf(exo_idx("ea"))
print(f"Tech: y[0]={irf_tech[0,y_i]:.4f}(+), yf[0]={irf_tech[0,yf_i]:.4f}(+), pi[0]={irf_tech[0,pi_i]:.4f}(-)")
print(f"Tech: gap[0]=y-yf={irf_tech[0,y_i]-irf_tech[0,yf_i]:.4f}(-exp)")

irf_mp = compute_irf(exo_idx("em"))
print(f"Monetary: r[0]={irf_mp[0,r_i]:.4f}(+), y[0]={irf_mp[0,y_i]:.4f}(-), pi[0]={irf_mp[0,pi_i]:.4f}(-)")

irf_pu = compute_irf(exo_idx("epinf"))
print(f"Price markup: pi[0]={irf_pu[0,pi_i]:.4f}(+), gap[0]={irf_pu[0,y_i]-irf_pu[0,yf_i]:.4f}(-)")

# ── Build embedded data for HTML ───────────────────────────────────────────
# We embed ghx, ghu, state_rows and variable indices so JS replicates IRFs.
# JS handles: multiple simultaneous shocks, HP filter, trend overlay.

js_data = {
    "ghx": ghx.tolist(),
    "ghu": ghu.tolist(),
    "state_rows": state_rows,       # 0-indexed
    "var_idx": {
        "y":   y_i,
        "yf":  yf_i,
        "pi":  pi_i,
        "r":   r_i,
        "rrf": rrf_i,
        "c":   c_i,
        "inv": inv_i,
        "lab": lab_i,
        "w":   w_i,
    },
    "shocks": shock_config,
    "params": params,
    "n_endo":   n_endo,
    "n_states": n_states,
    "n_exo":    n_exo,
}

js_data_json = json.dumps(js_data, separators=(',', ':'))

# ── Generate HTML ──────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Smets-Wouters 2007 — Impulse Response Explorer</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

:root {{
  --bg:      #ffffff; --bg2: #f5f4f0; --bg3: #eeece8;
  --border:  rgba(0,0,0,0.10); --border2: rgba(0,0,0,0.18);
  --text:    #1a1a18; --text2: #5f5e5a; --text3: #888780;
  --blue:    #378ADD; --teal: #1D9E75; --coral: #D85A30;
  --amber:   #BA7517; --purple: #7C55C8;
  --r-md: 8px; --r-lg: 12px;
}}
@media (prefers-color-scheme: dark) {{
  :root {{
    --bg: #1c1c1a; --bg2: #252523; --bg3: #2e2e2b;
    --border: rgba(255,255,255,0.09); --border2: rgba(255,255,255,0.18);
    --text: #e8e6df; --text2: #b4b2a9; --text3: #888780;
  }}
}}

body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
       background: var(--bg3); color: var(--text); font-size: 14px; line-height: 1.5; }}

.page {{ max-width: 1200px; margin: 0 auto; padding: 2rem 1.5rem 4rem; }}

h1 {{ font-size: 20px; font-weight: 500; margin-bottom: 4px; }}
.subtitle {{ font-size: 12px; color: var(--text2); margin-bottom: 1.5rem; }}

.card {{ background: var(--bg); border: 0.5px solid var(--border);
        border-radius: var(--r-lg); padding: 1.1rem 1.25rem; margin-bottom: 1rem; }}
.card-title {{ font-size: 11px; font-weight: 500; color: var(--text3);
               text-transform: uppercase; letter-spacing: .06em; margin-bottom: .85rem; }}

/* Shock grid */
.shock-grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(220px, 1fr)); gap: 10px; }}
.shock-row  {{ display: flex; align-items: center; gap: 10px; background: var(--bg2);
               border-radius: var(--r-md); padding: 9px 12px; }}
.shock-toggle {{ position: relative; width: 34px; height: 19px; flex-shrink: 0; cursor: pointer; }}
.shock-toggle input {{ opacity: 0; width: 0; height: 0; }}
.shock-track {{ position: absolute; inset: 0; background: var(--border2);
                border-radius: 10px; transition: .2s; }}
.shock-toggle input:checked + .shock-track {{ background: var(--blue); }}
.shock-thumb {{ position: absolute; width: 13px; height: 13px; background: white;
                border-radius: 50%; top: 3px; left: 3px; transition: .2s; }}
.shock-toggle input:checked ~ .shock-thumb {{ left: 18px; }}
.shock-label {{ flex: 1; font-size: 13px; color: var(--text); }}
.shock-size  {{ width: 52px; font-size: 12px; text-align: right; color: var(--text2); }}

.shock-slider-wrap {{ margin-top: 6px; }}
.shock-slider-wrap input[type=range] {{ width: 100%; margin-top: 4px; }}

/* Sliders */
input[type=range] {{ -webkit-appearance: none; height: 3px; border-radius: 2px;
                     background: var(--bg3); outline: none; cursor: pointer; }}
input[type=range]::-webkit-slider-thumb {{ -webkit-appearance: none; width: 14px; height: 14px;
  border-radius: 50%; background: var(--blue); border: 2px solid var(--bg); }}
input[type=range]::-moz-range-thumb {{ width: 12px; height: 12px; border-radius: 50%; background: var(--blue); border: 2px solid var(--bg); }}

/* Controls row */
.ctrl-row {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 10px; }}
.ctrl {{ background: var(--bg2); border-radius: var(--r-md); padding: 10px 12px; }}
.ctrl-head {{ display: flex; justify-content: space-between; margin-bottom: 6px; }}
.ctrl-name {{ font-size: 12px; color: var(--text2); }}
.ctrl-val  {{ font-size: 14px; font-weight: 500; color: var(--text); font-variant-numeric: tabular-nums; }}

/* Legend */
.legend {{ display: flex; gap: 18px; flex-wrap: wrap; font-size: 12px;
           color: var(--text2); margin-bottom: .85rem; }}
.leg {{ display: flex; align-items: center; gap: 5px; }}
.leg-line {{ width: 20px; height: 2px; border-radius: 1px; flex-shrink: 0; }}
.leg-line.dashed {{ background: none !important; border-top: 2px dashed; }}

/* Chart grid */
.chart-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(310px, 1fr)); gap: 12px; }}
.chart-card {{ background: var(--bg); border: 0.5px solid var(--border);
               border-radius: var(--r-lg); padding: 12px 14px; }}
.chart-name {{ font-size: 13px; font-weight: 500; color: var(--text); margin-bottom: 2px; }}
.chart-sub  {{ font-size: 11px; color: var(--text3); margin-bottom: 8px; }}
.chart-wrap {{ position: relative; height: 190px; }}

/* Equation footer */
.eq {{ font-size: 11px; color: var(--text3); line-height: 2; }}

@media (max-width: 600px) {{
  .page {{ padding: 1rem; }}
  .chart-grid {{ grid-template-columns: 1fr; }}
}}
</style>
</head>
<body>
<div class="page">

<h1>Smets-Wouters (2007) — Impulse Response Explorer</h1>
<p class="subtitle">
  Full SW model solved via Dynare. Select any combination of shocks, set sizes and persistence,
  and see IRFs for output, potential, inflation and rates over up to 80 quarters.
  Trend growth ({params['ctrend']:.2f}%/qtr) and inflation target ({params['constepinf']:.2f}%) added to level charts.
</p>

<!-- Shock selector -->
<div class="card">
  <div class="card-title">Shocks — select any combination</div>
  <div class="shock-grid" id="shock-grid"></div>
</div>

<!-- Shock persistence + horizon -->
<div class="card">
  <div class="card-title">Simulation settings</div>
  <div class="ctrl-row">
    <div class="ctrl">
      <div class="ctrl-head">
        <span class="ctrl-name">Time horizon (quarters)</span>
        <span class="ctrl-val" id="v-T">40</span>
      </div>
      <input type="range" id="sl-T" min="20" max="80" step="5" value="40" oninput="update()">
    </div>
  </div>
</div>

<!-- Legend -->
<div class="legend">
  <div class="leg"><div class="leg-line" style="background:var(--blue)"></div>NK output (actual)</div>
  <div class="leg"><div class="leg-line dashed" style="border-color:var(--teal)"></div>Flex-price output (y*)</div>
  <div class="leg"><div class="leg-line dashed" style="border-color:var(--coral)"></div>HP-filtered trend</div>
  <div class="leg"><div class="leg-line" style="background:var(--amber)"></div>Natural rate (r*)</div>
</div>

<!-- Charts -->
<div class="chart-grid">
  <div class="chart-card">
    <div class="chart-name">Output (log level, %)</div>
    <div class="chart-sub">NK output + trend; flex-price y* dashed; HP trend dotted</div>
    <div class="chart-wrap"><canvas id="c-y" aria-label="Output IRF">Output chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Output gap (y − y*)</div>
    <div class="chart-sub">NK output minus flex-price counterfactual</div>
    <div class="chart-wrap"><canvas id="c-gap" aria-label="Output gap IRF">Gap chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Inflation (π)</div>
    <div class="chart-sub">Annualised; baseline = {params['constepinf']:.1f}% target</div>
    <div class="chart-wrap"><canvas id="c-pi" aria-label="Inflation IRF">Inflation chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Nominal rate (r)</div>
    <div class="chart-sub">Annualised; deviation from steady state</div>
    <div class="chart-wrap"><canvas id="c-r" aria-label="Nominal rate IRF">Rate chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Natural rate (r*)</div>
    <div class="chart-sub">Flex-price real interest rate (annualised)</div>
    <div class="chart-wrap"><canvas id="c-rnat" aria-label="Natural rate IRF">Natural rate chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Consumption (c)</div>
    <div class="chart-sub">Log deviation + trend</div>
    <div class="chart-wrap"><canvas id="c-c" aria-label="Consumption IRF">Consumption chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Investment (i)</div>
    <div class="chart-sub">Log deviation + trend</div>
    <div class="chart-wrap"><canvas id="c-inv" aria-label="Investment IRF">Investment chart</canvas></div>
  </div>
  <div class="chart-card">
    <div class="chart-name">Hours worked (l) &amp; Real wage (w)</div>
    <div class="chart-sub">Log deviations</div>
    <div class="chart-wrap"><canvas id="c-lab" aria-label="Labour IRF">Labour chart</canvas></div>
  </div>
</div>

<!-- Model info -->
<div class="card" style="margin-top:1rem">
  <div class="card-title">Model — Smets &amp; Wouters (2007)</div>
  <div class="eq">
    Households: habits λ={params['chabb']:.2f} · risk aversion σ={params['csigma']:.2f}<br>
    Prices: Calvo ξ_p={params['cprobp']:.2f} · Wages: Calvo ξ_w={params['cprobw']:.2f}<br>
    Capital accumulation · investment adjustment costs · capital utilization<br>
    Taylor rule: ρ={params['crr']:.2f}, φ_π={params['crpi']:.2f}, φ_y={params['cry']:.2f}, φ_Δy={params['crdy']:.2f}<br>
    Trend growth: {params['ctrend']:.3f}%/qtr ({params['ctrend']*4:.2f}%/yr) ·
    Inflation target: {params['constepinf']:.2f}%/qtr ({params['constepinf']*4:.2f}%/yr) ·
    Solution: Dynare first-order perturbation (ghx, ghu exported)
  </div>
</div>

</div><!-- .page -->

<script>
// ── Embedded model solution ────────────────────────────────────────────────
const MODEL = {js_data_json};

const GHX        = MODEL.ghx;           // n_endo x n_states
const GHU        = MODEL.ghu;           // n_endo x n_exo
const STATE_ROWS = MODEL.state_rows;    // which rows are states (0-indexed)
const VI         = MODEL.var_idx;       // variable indices (0-indexed)
const SHOCKS     = MODEL.shocks;        // array of {{key,label,idx}}
const P          = MODEL.params;
const N_ENDO     = MODEL.n_endo;
const N_STATES   = MODEL.n_states;

const TREND_Q    = P.ctrend;            // % per quarter
const PI_TARGET  = P.constepinf;       // % per quarter

// ── HP filter (exact matrix solve, Hodrick-Prescott λ=1600) ───────────────
function hpFilter(y, lam = 1600) {{
  const n = y.length;
  // Build (I + λ D'D) as dense matrix and solve
  // D'D is pentadiagonal; for n≤80 dense is fine
  const A = Array.from({{length: n}}, (_, i) => new Array(n).fill(0));
  for (let i = 0; i < n; i++) A[i][i] = 1;
  for (let i = 2; i < n; i++) {{
    const pairs = [[i,i,6],[i,i-1,-4],[i,i-2,1],[i-1,i,-4],[i-2,i,1],[i-1,i-1,4]];
    // Actually build D'D correctly via the second-difference matrix
  }}
  // Build D (n-2 x n second difference matrix), then D'D
  const D = Array.from({{length: n-2}}, () => new Array(n).fill(0));
  for (let i = 0; i < n-2; i++) {{
    D[i][i]=1; D[i][i+1]=-2; D[i][i+2]=1;
  }}
  // A = I + lam * D'D
  const DtD = Array.from({{length: n}}, () => new Array(n).fill(0));
  for (let i = 0; i < n-2; i++)
    for (let j = 0; j < n; j++)
      for (let k = 0; k < n; k++)
        DtD[j][k] += D[i][j] * D[i][k];
  for (let i = 0; i < n; i++) {{
    A[i][i] = 1;
    for (let j = 0; j < n; j++) A[i][j] += lam * DtD[i][j];
  }}
  // Gaussian elimination
  const aug = A.map((row, i) => [...row, y[i]]);
  for (let col = 0; col < n; col++) {{
    let mx = col;
    for (let r = col+1; r < n; r++) if (Math.abs(aug[r][col]) > Math.abs(aug[mx][col])) mx = r;
    [aug[col], aug[mx]] = [aug[mx], aug[col]];
    const piv = aug[col][col];
    if (Math.abs(piv) < 1e-14) continue;
    for (let r = col+1; r < n; r++) {{
      const f = aug[r][col]/piv;
      for (let k = col; k <= n; k++) aug[r][k] -= f*aug[col][k];
    }}
  }}
  const trend = new Array(n).fill(0);
  for (let i = n-1; i >= 0; i--) {{
    let s = aug[i][n];
    for (let j = i+1; j < n; j++) s -= aug[i][j]*trend[j];
    trend[i] = s/aug[i][i];
  }}
  return trend;
}}

// ── IRF propagation ────────────────────────────────────────────────────────
function propagate(shockVec, T) {{
  // shockVec: length n_exo, the impact shock
  // Returns (T x n_endo) array
  const path = Array.from({{length: T}}, () => new Array(N_ENDO).fill(0));
  // Impact period
  for (let v = 0; v < N_ENDO; v++) {{
    let s = 0;
    for (let e = 0; e < GHU[v].length; e++) s += GHU[v][e] * shockVec[e];
    path[0][v] = s;
  }}
  // Propagate
  for (let t = 1; t < T; t++) {{
    const stateVec = STATE_ROWS.map(r => path[t-1][r]);
    for (let v = 0; v < N_ENDO; v++) {{
      let s = 0;
      for (let k = 0; k < N_STATES; k++) s += GHX[v][k] * stateVec[k];
      path[t][v] = s;
    }}
  }}
  return path;
}}

// Combine multiple shocks (superposition — valid under linearity)
function computeIRF(shockEntries, T) {{
  // shockEntries: [{{shockIdx, size, persistence}}, ...]
  // Build combined path by summing individual paths (scaled by size, adjusted for persistence)
  const combined = Array.from({{length: T}}, () => new Array(N_ENDO).fill(0));

  for (const entry of shockEntries) {{
    const {{shockIdx, size, persistence}} = entry;
    // Impact vector scaled by size
    const impactVec = new Array(GHU[0].length).fill(0);
    impactVec[shockIdx] = size;
    // To incorporate user-set persistence: the AR(1) shock in the model already
    // has its own persistence from the estimated rho. To let the user override
    // persistence, we re-scale future impacts by (user_rho / model_rho)^t.
    // This is an approximation — the cleanest would be to re-solve the model,
    // but for display purposes this gives intuitive behaviour.
    // For t=0 the impact is the same; for t>0 we scale by cumulative rho ratio.
    const modelRho = getModelRho(entry.key);
    const path0 = propagate(impactVec, T);

    for (let t = 0; t < T; t++) {{
      const scale = t === 0 ? 1.0 : Math.pow(persistence / Math.max(modelRho, 0.01), t);
      for (let v = 0; v < N_ENDO; v++) combined[t][v] += path0[t][v] * scale;
    }}
  }}
  return combined;
}}

function getModelRho(key) {{
  const rhos = {{
    ea: 0.9977, eb: 0.5799, eg: 0.9957,
    eqs: 0.7165, em: 0.0, epinf: 0.0, ew: 0.0
  }};
  return rhos[key] || 0.5;
}}

// ── Chart setup ────────────────────────────────────────────────────────────
const isDark = () => window.matchMedia('(prefers-color-scheme: dark)').matches;

function mkDataset(data, color, dash=[], width=1.8, label='') {{
  return {{data, label, borderColor: color, borderWidth: width,
           borderDash: dash, pointRadius: 0, tension: 0.3, fill: false}};
}}

function mkChart(id, datasets) {{
  const dark = isDark();
  const gc = dark ? 'rgba(255,255,255,0.07)' : 'rgba(0,0,0,0.06)';
  const tc = dark ? '#777' : '#999';
  return new Chart(document.getElementById(id), {{
    type: 'line',
    data: {{ labels: [], datasets }},
    options: {{
      responsive: true, maintainAspectRatio: false,
      animation: {{ duration: 200 }},
      interaction: {{ mode: 'index', intersect: false }},
      plugins: {{
        legend: {{ display: false }},
        tooltip: {{ callbacks: {{ label: c => `${{c.dataset.label||''}}: ${{c.parsed.y.toFixed(3)}}` }} }}
      }},
      scales: {{
        x: {{ grid: {{ color: gc }}, ticks: {{ color: tc, font: {{size:10}}, maxTicksLimit: 9 }},
              title: {{ display: true, text: 'quarters', color: tc, font: {{size:10}} }} }},
        y: {{ grid: {{
                color: ctx => ctx.tick.value === 0
                  ? (dark ? 'rgba(255,255,255,0.2)' : 'rgba(0,0,0,0.15)')
                  : gc,
                lineWidth: ctx => ctx.tick.value === 0 ? 1.5 : 1
              }},
              ticks: {{ color: tc, font: {{size:10}}, callback: v => v.toFixed(3) }} }}
      }}
    }}
  }});
}}

const C = {{
  blue: '#378ADD', teal: '#1D9E75', coral: '#D85A30', amber: '#BA7517', purple: '#7C55C8'
}};

const charts = {{
  y:   mkChart('c-y',   [mkDataset([],C.blue,[],1.8,'NK output'),
                          mkDataset([],C.teal,[5,4],1.4,'Flex-price y*'),
                          mkDataset([],C.coral,[3,3],1.2,'HP trend')]),
  gap: mkChart('c-gap', [mkDataset([],C.blue,[],1.8,'Output gap y−y*')]),
  pi:  mkChart('c-pi',  [mkDataset([],C.blue,[],1.8,'Inflation'),
                          mkDataset([],C.teal,[5,4],1.2,'Target')]),
  r:   mkChart('c-r',   [mkDataset([],C.blue,[],1.8,'Nominal rate'),
                          mkDataset([],C.amber,[5,4],1.2,'Natural rate (flex)')]),
  rnat:mkChart('c-rnat',[mkDataset([],C.amber,[],1.8,'Natural rate r*')]),
  c:   mkChart('c-c',   [mkDataset([],C.blue,[],1.8,'Consumption')]),
  inv: mkChart('c-inv', [mkDataset([],C.blue,[],1.8,'Investment')]),
  lab: mkChart('c-lab', [mkDataset([],C.blue,[],1.8,'Hours worked'),
                          mkDataset([],C.purple,[5,4],1.4,'Real wage')]),
}};

// ── Shock controls ─────────────────────────────────────────────────────────
const shockState = {{}};
SHOCKS.forEach(sh => {{
  shockState[sh.key] = {{ active: false, size: 1.0, rho: getModelRho(sh.key) }};
}});

function buildShockGrid() {{
  const grid = document.getElementById('shock-grid');
  SHOCKS.forEach(sh => {{
    const div = document.createElement('div');
    div.className = 'shock-row';
    div.innerHTML = `
      <label class="shock-toggle">
        <input type="checkbox" id="chk-${{sh.key}}" onchange="toggleShock('${{sh.key}}')">
        <div class="shock-track"></div>
        <div class="shock-thumb"></div>
      </label>
      <span class="shock-label">${{sh.label}}</span>
      <span class="shock-size" id="sz-${{sh.key}}">1.0σ</span>
    `;
    // Size slider below
    const sliderWrap = document.createElement('div');
    sliderWrap.className = 'shock-slider-wrap';
    sliderWrap.style.cssText = 'grid-column: 1/-1; padding: 0 12px 9px;';
    sliderWrap.innerHTML = `
      <div style="display:flex;justify-content:space-between;font-size:11px;color:var(--text3);margin-bottom:3px">
        <span>Size</span><span id="sz2-${{sh.key}}">1.0 σ</span>
      </div>
      <input type="range" min="0.1" max="3" step="0.1" value="1.0"
             id="sl-size-${{sh.key}}"
             oninput="setSize('${{sh.key}}', this.value)">
      <div style="display:flex;justify-content:space-between;font-size:11px;color:var(--text3);margin-top:6px;margin-bottom:3px">
        <span>Persistence ρ</span><span id="rho-${{sh.key}}">${{getModelRho(sh.key).toFixed(2)}}</span>
      </div>
      <input type="range" min="0" max="0.98" step="0.01" value="${{getModelRho(sh.key)}}"
             id="sl-rho-${{sh.key}}"
             oninput="setRho('${{sh.key}}', this.value)">
    `;
    // Wrap both in a sub-grid
    const outer = document.createElement('div');
    outer.style.cssText = 'background:var(--bg2);border-radius:var(--r-md);overflow:hidden;';
    outer.appendChild(div);
    outer.appendChild(sliderWrap);
    grid.appendChild(outer);
  }});
}}

function toggleShock(key) {{
  shockState[key].active = document.getElementById(`chk-${{key}}`).checked;
  update();
}}
function setSize(key, val) {{
  shockState[key].size = parseFloat(val);
  document.getElementById(`sz-${{key}}`).textContent = parseFloat(val).toFixed(1) + 'σ';
  document.getElementById(`sz2-${{key}}`).textContent = parseFloat(val).toFixed(1) + ' σ';
  update();
}}
function setRho(key, val) {{
  shockState[key].rho = parseFloat(val);
  document.getElementById(`rho-${{key}}`).textContent = parseFloat(val).toFixed(2);
  update();
}}

// ── Main update ────────────────────────────────────────────────────────────
function update() {{
  const T = parseInt(document.getElementById('sl-T').value);
  document.getElementById('v-T').textContent = T;

  const activeShocks = SHOCKS
    .filter(sh => shockState[sh.key].active)
    .map(sh => ({{
      key:      sh.key,
      shockIdx: sh.idx,
      size:     shockState[sh.key].size,
      persistence: shockState[sh.key].rho,
    }}));

  const quarters = Array.from({{length: T}}, (_, i) => i);

  // If no shocks active, show zeros
  if (activeShocks.length === 0) {{
    Object.values(charts).forEach(ch => {{
      ch.data.labels = quarters;
      ch.data.datasets.forEach(ds => {{ ds.data = new Array(T).fill(0); }});
      ch.update('none');
    }});
    return;
  }}

  const path = computeIRF(activeShocks, T);

  // Extract series (log-deviations, already in %)
  const y_dev   = path.map(s => s[VI.y]);
  const yf_dev  = path.map(s => s[VI.yf]);
  const pi_dev  = path.map(s => s[VI.pi]);
  const r_dev   = path.map(s => s[VI.r]);
  const rrf_dev = path.map(s => s[VI.rrf]);
  const c_dev   = path.map(s => s[VI.c]);
  const inv_dev = path.map(s => s[VI.inv]);
  const lab_dev = path.map(s => s[VI.lab]);
  const w_dev   = path.map(s => s[VI.w]);

  const gap   = y_dev.map((v, i) => v - yf_dev[i]);

  // Level paths: add deterministic trend (in log %)
  // In log space: Y_t = exp(y_dev_t) * exp(trend*t) * Y_0
  // Since we display as % deviation from trend, levels are just dev + trend*t
  const trendPath = quarters.map(t => TREND_Q * t);

  const y_level  = y_dev.map((v, t) => v + trendPath[t]);
  const yf_level = yf_dev.map((v, t) => v + trendPath[t]);
  const hp_trend = hpFilter(y_level);

  // Inflation: annualised quarters → multiply by 4; add target
  const pi_ann    = pi_dev.map(v => v * 4 + PI_TARGET * 4);
  const target    = new Array(T).fill(PI_TARGET * 4);
  const r_ann     = r_dev.map(v => v * 4);
  const rrf_ann   = rrf_dev.map(v => v * 4);
  const c_level   = c_dev.map((v, t) => v + trendPath[t]);
  const inv_level = inv_dev.map((v, t) => v + trendPath[t]);

  function setDs(chart, idx, data) {{ chart.data.datasets[idx].data = data; }}
  function setLabels(chart) {{ chart.data.labels = quarters; }}

  Object.values(charts).forEach(ch => setLabels(ch));

  // Output level chart
  setDs(charts.y, 0, y_level);
  setDs(charts.y, 1, yf_level);
  setDs(charts.y, 2, hp_trend);

  // Gap
  setDs(charts.gap, 0, gap);

  // Inflation
  setDs(charts.pi, 0, pi_ann);
  setDs(charts.pi, 1, target);

  // Nominal rate
  setDs(charts.r, 0, r_ann);
  setDs(charts.r, 1, rrf_ann);

  // Natural rate
  setDs(charts.rnat, 0, rrf_ann);

  // Consumption
  setDs(charts.c, 0, c_level);

  // Investment
  setDs(charts.inv, 0, inv_level);

  // Labour
  setDs(charts.lab, 0, lab_dev);
  setDs(charts.lab, 1, w_dev);

  Object.values(charts).forEach(ch => ch.update('none'));
}}

// ── Initialise ─────────────────────────────────────────────────────────────
document.getElementById('sl-T').addEventListener('input', update);
buildShockGrid();
update();
</script>
</body>
</html>"""

with open("visual.html", "w", encoding="utf-8") as f:
    f.write(html)

print("\n✅ Generated visual.html")
print("   Open directly in browser or push to GitHub Pages.")
print("   No server needed — fully self-contained.")

ghx: (40, 20), ghu: (40, 7)
State rows (0-idx): [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]

Sanity checks:
Tech: y[0]=0.7794(+), yf[0]=1.2300(+), pi[0]=-0.1338(-)
Tech: gap[0]=y-yf=-0.4505(-exp)
Monetary: r[0]=0.6577(+), y[0]=-1.2277(-), pi[0]=-0.2453(-)
Price markup: pi[0]=1.1767(+), gap[0]=-0.4624(-)

✅ Generated visual.html
   Open directly in browser or push to GitHub Pages.
   No server needed — fully self-contained.
